# Sprint 5

## Install PySpark

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 12:55:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Shuffle partitions: 8


## Import the funtions and create data path

In [3]:
from pathlib import Path

from pyspark.sql.window import Window
import pyspark.sql.functions as F 
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    avg,
    count,
    count_distinct,
    broadcast,
    substring,
    min as spark_min,
    max as spark_max,
    mean,
    approx_percentile
)

DATA_DIR = Path("../data/MIMIC-IV/hosp")
GENERAL_DATA_DIR = Path("../data")
EVIDENCE_DIR = Path("../out/evidence")
VISIT_DF = Path("../data/visits")
DIAGNOSES_DF = Path("../data/diagnoses")
ICD_CODES_DF = Path("../data/icd_codes")

## Dataframes

In [4]:
# -------------------------------------------------------
# Pulling Dataframes from Saved Parquet Files
# -------------------------------------------------------
visits = spark.read.parquet(str(VISIT_DF))
visits.show(10, truncate=False)

diagnoses = spark.read.parquet(str(DIAGNOSES_DF))
diagnoses.show(10, truncate=False)

icd_codes = spark.read.parquet(str(ICD_CODES_DF))
icd_codes.show(10, truncate=False)



+----------+--------+----------------------+------+-----------+---+----------+
|subject_id|hadm_id |race                  |gender|visit_type |age|admit_day |
+----------+--------+----------------------+------+-----------+---+----------+
|10001401  |21544441|WHITE                 |F     |BC_FIRST_DX|89 |2014-06-04|
|10015568  |26581506|BLACK/AFRICAN         |M     |BC_FIRST_DX|65 |2011-08-19|
|10024451  |22358047|WHITE                 |M     |BC_FIRST_DX|70 |2020-09-14|
|10024483  |27517184|BLACK/AFRICAN AMERICAN|M     |BC_FIRST_DX|82 |2008-07-03|
|10026950  |28254249|WHITE                 |M     |BC_FIRST_DX|91 |2011-03-14|
|10068474  |25255224|WHITE                 |F     |BC_FIRST_DX|71 |2017-02-07|
|10070928  |26961908|WHITE                 |M     |BC_FIRST_DX|87 |2008-04-05|
|10085948  |28355680|WHITE                 |F     |BC_FIRST_DX|39 |2017-04-04|
|10098814  |23431301|WHITE                 |F     |BC_FIRST_DX|69 |2020-08-11|
|10099497  |28250562|WHITE                 |M     |B

In [10]:
# Finding out the top Relevant symptoms from each
# Making sure to have one output that merges

#defining a window for the top ranked row to only use that per hadm_id

print( diagnoses.groupBy("subject_id"))

top_symptoms = (
    diagnoses
    .filter( col("visit_type")=="SYMPTOM")
    .join(
        broadcast(icd_codes.filter(col("status") == "RELEVANT")), #broadcast join on filtered dataset
        on=["icd_code","icd_version"],
        how="inner"
    )
    .where(col("ranking") <= 10)
    .withColumn("cons_icd_code",
         when(       )

    )
    .groupBy("icd_code") # decided to just go for top 10, will figure out
    .agg(
        count("*").alias("total_count"),
        count_distinct("subject_id").alias("subject_count")
)
    .sort("total_count", ascending=False)
)

top_symptoms.show(10, truncate=False)


DataFrame[subject_id: int, count: bigint]
+--------+-----------+-------------+
|icd_code|total_count|subject_count|
+--------+-----------+-------------+
|5990    |105        |68           |
|N390    |85         |63           |
|78820   |34         |28           |
|R319    |31         |29           |
|59970   |21         |20           |
|59971   |17         |17           |
|R339    |16         |14           |
|R310    |15         |13           |
|N329    |15         |14           |
|R338    |11         |9            |
+--------+-----------+-------------+
only showing top 10 rows


## Clean up
Stop spark session when done

In [5]:
# Uncomment when you are completely done:

spark.stop()